# 28 · 多工具 + 长期记忆 + 预算控制

> **学习目标**：把入门 Agent 升级成「**像样的生产骨架**」—— 4 个工具 + 长期记忆（向量库召回历史）+ 步数/token 双预算 + 软停。
>
> **预备**：25 / 26 / 27 跑过。01-RAG 09 号 (mini_vecdb) 见过。
>
> **为什么重要**：生产 Agent 的稳定性 90% 来自这 3 件事 ——「能调多种工具且不混 / 记得住历史 / 不会跑飞」。

In [1]:
MODE = 'OFFLINE'

import re, json, uuid, time, hashlib, requests
import numpy as np
from pathlib import Path
from datetime import datetime
OLLAMA = 'http://127.0.0.1:11434'

def fake_embed(text, dim=128):
    seed = int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)
    v = np.random.default_rng(seed).standard_normal(dim).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

# 简易 token 估算（中文 ~1 char/token，英文 ~4 char/token；粗糙但够用）
def est_tokens(text: str) -> int:
    zh = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')
    other = len(text) - zh
    return zh + other // 4 + 1

print(f'MODE = {MODE}')

MODE = OFFLINE


D:\ProgramData\anaconda3\envs\rag\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


## 1. 4 个工具 —— 真正多样化的能力组合

**关键设计**：副作用工具 vs 只读工具区分；输入用 JSON schema；错误返回结构化。

In [2]:
# 沙箱目录（不污染外部）
SANDBOX = Path('./_agent_sandbox').resolve()
SANDBOX.mkdir(exist_ok=True)
(SANDBOX / 'notes.txt').write_text('已存的笔记：\n- python\n- rag\n', encoding='utf-8')
(SANDBOX / 'todo.txt').write_text('TODO: 写 28 号 notebook 完成 02-Agent\n', encoding='utf-8')

def tool_read_file(path: str) -> str:
    p = SANDBOX / path
    if not p.is_file():
        return json.dumps({'error': f'file not found: {path}'}, ensure_ascii=False)
    return p.read_text(encoding='utf-8')

def tool_write_file(path: str, content: str) -> str:
    p = SANDBOX / path
    if '..' in path or path.startswith('/'):
        return json.dumps({'error': 'path escape blocked'}, ensure_ascii=False)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')
    return json.dumps({'ok': True, 'bytes': len(content.encode('utf-8'))}, ensure_ascii=False)

def tool_search_web(query: str) -> str:
    """mock：根据 query 关键词返回固定结果。生产换 Tavily / Bing API。"""
    KB = {
        'python': ['Python 是动态类型的解释型语言', 'Python 创始人 Guido van Rossum'],
        'rag':    ['RAG = 检索增强生成', 'RAG 流程: load→chunk→embed→retrieve→generate'],
        'qwen':   ['Qwen 是阿里通义千问系列', 'Qwen2.5 系列含 0.5B/1.5B/3B/7B/14B/72B'],
    }
    q_lower = query.lower()
    hits = [v for k, vs in KB.items() if k in q_lower for v in vs]
    return json.dumps({'query': query, 'results': hits or ['no results']}, ensure_ascii=False)

# 第 4 个工具 在第 2 节定义（recall_memory）—— 因为需要 memory store

# Smoke
print(tool_read_file('notes.txt'))
print(tool_write_file('out.txt', '测试写入'))
print(tool_search_web('什么是 RAG'))

已存的笔记：
- python
- rag

{"ok": true, "bytes": 12}
{"query": "什么是 RAG", "results": ["RAG = 检索增强生成", "RAG 流程: load→chunk→embed→retrieve→generate"]}


## 2. 长期记忆 —— 向量库召回历史对话

**思路**：每次 Agent 跑完，把 `(query, final_answer)` 存入向量库（embed query 作 key）。新 query 进来 → 先召回 top-2 历史 → 拼进 system prompt 作「之前类似问题的处理」。

**与 RAG 的关系**：本质就是 RAG，但「文档」是 Agent 自己的历史。

In [3]:
class LongTermMemory:
    """50 行内的向量库记忆（沿用 01-RAG 09 号思路）。"""
    def __init__(self):
        self.records = []   # [{'id', 'query', 'answer', 'vec', 'ts'}]
        self.vectors = None

    def remember(self, query: str, answer: str):
        vec = fake_embed(query)
        self.records.append({'id': uuid.uuid4().hex[:8], 'query': query, 'answer': answer,
                              'vec': vec, 'ts': datetime.now().isoformat()})
        self.vectors = np.vstack([r['vec'] for r in self.records])

    def recall(self, query: str, top_k: int = 2) -> list[dict]:
        if not self.records:
            return []
        sims = self.vectors @ fake_embed(query)
        order = np.argsort(-sims)[:top_k]
        return [{'query': self.records[i]['query'], 'answer': self.records[i]['answer'],
                  'sim': float(sims[i])} for i in order if sims[i] > 0.3]

memory = LongTermMemory()

def tool_recall_memory(query: str) -> str:
    hits = memory.recall(query, top_k=3)
    return json.dumps({'query': query, 'recalled': hits}, ensure_ascii=False)

# 4 工具齐了
TOOLS = {
    'read_file':       {'fn': tool_read_file,       'side_effect': False, 'desc': '读沙箱里的文件。参数: path (str)'},
    'write_file':      {'fn': tool_write_file,      'side_effect': True,  'desc': '写沙箱里的文件。参数: path (str), content (str)'},
    'search_web':      {'fn': tool_search_web,      'side_effect': False, 'desc': '搜 web (mock)。参数: query (str)'},
    'recall_memory':   {'fn': tool_recall_memory,   'side_effect': False, 'desc': '从 Agent 长期记忆里召回类似历史。参数: query (str)'},
}
print(f'共 {len(TOOLS)} 工具')

共 4 工具


## 3. 预算控制 —— 步数 / token / 时间三路硬刹

**为什么必要**：LLM 偶尔会陷入「调同一工具不同输入 10 次都失败」的循环。**没预算 = 烧光配额**。

**3 种预算**：
- `max_iter` —— 最重要，硬上限
- `token_budget` —— 累计 input/output token 超过就停
- `time_budget_s` —— 钟表时间超过就停

In [4]:
class Budget:
    def __init__(self, max_iter=8, token_budget=2000, time_budget_s=30):
        self.max_iter = max_iter
        self.token_budget = token_budget
        self.time_budget_s = time_budget_s
        self.tokens_used = 0
        self.t_start = time.perf_counter()
        self.iter = 0

    def tick(self, llm_in: str, llm_out: str) -> tuple[bool, str]:
        self.iter += 1
        self.tokens_used += est_tokens(llm_in) + est_tokens(llm_out)
        elapsed = time.perf_counter() - self.t_start
        if self.iter > self.max_iter:
            return False, f'达到 max_iter={self.max_iter}'
        if self.tokens_used > self.token_budget:
            return False, f'达到 token_budget={self.token_budget}（已用 {self.tokens_used}）'
        if elapsed > self.time_budget_s:
            return False, f'达到 time_budget={self.time_budget_s}s'
        return True, ''

    def report(self) -> dict:
        return {
            'iter': self.iter, 'tokens_used': self.tokens_used,
            'time_s': round(time.perf_counter() - self.t_start, 3),
        }

## 4. Agent loop —— 集成 4 工具 + 记忆 + 预算

**关键流程**：
1. 新 query 进来 → **先自动 recall** 长期记忆（不让 LLM 决定，由系统强制做）
2. 把 recall 结果作 `[历史参考]` 拼进 system prompt
3. 跑 Agent loop（同 25 号的 ReAct）
4. 每步 `budget.tick()`，超就软停
5. 跑完把 `(query, answer)` 存进长期记忆

In [5]:
def llm_offline(messages: list[dict]) -> str:
    """按规则模拟。本 notebook 主要演示工具组合 + 预算，LLM stub 极简。"""
    user_q = next((m['content'] for m in reversed(messages) if m['role'] == 'user'), '')
    transcript = '\n'.join(m['content'] for m in messages if m['role'] == 'assistant')
    n_obs = transcript.count('Observation:')

    # 启发式：先决定工具
    if '写入' in user_q or '保存' in user_q:
        if n_obs == 0:
            content = re.search(r'内容[：:](.+?)(?:到|$)', user_q)
            path = re.search(r'到\s*(\S+\.txt)', user_q)
            return (f'Thought: 写入文件\nAction: write_file\n'
                    f'Action Input: {json.dumps({"path": (path.group(1) if path else "new.txt"), "content": content.group(1).strip() if content else user_q[:50]}, ensure_ascii=False)}')
    if '搜' in user_q or '查' in user_q:
        if n_obs == 0:
            kw = re.search(r'(python|rag|qwen)', user_q.lower())
            if kw:
                return f'Thought: 搜 web\nAction: search_web\nAction Input: {json.dumps({"query": kw.group(1)})}'
    if '读' in user_q or '内容' in user_q:
        if n_obs == 0:
            m = re.search(r'(\w+\.txt)', user_q)
            if m:
                return f'Thought: 读文件\nAction: read_file\nAction Input: {json.dumps({"path": m.group(1)})}'

    # 已有 observation → 综合给答案
    last_obs = re.findall(r'Observation:\s*(.*)', transcript)
    if last_obs:
        return f'Thought: 已得到结果\nFinal Answer: {last_obs[-1][:200]}'
    return f'Thought: 不需要工具\nFinal Answer: 对 "{user_q}" 我没有更多信息'

llm = llm_offline   # ONLINE 时换成 ollama，结构同 25 号

def parse(text: str) -> dict:
    fa = re.search(r'Final Answer:\s*(.*)', text, re.DOTALL)
    if fa: return {'type': 'final', 'answer': fa.group(1).strip()}
    am = re.search(r'Action:\s*(\S+)', text); im = re.search(r'Action Input:\s*(.*?)(?:\n|$)', text, re.DOTALL)
    if am and im:
        try: args = json.loads(im.group(1).strip())
        except json.JSONDecodeError: args = {'_raw': im.group(1).strip()}
        return {'type': 'action', 'tool': am.group(1), 'args': args}
    return {'type': 'error', 'raw': text}

def run_agent_v2(question: str, tools: dict, budget: Budget, memory: LongTermMemory, verbose: bool = True) -> dict:
    # 自动召回长期记忆
    recalled = memory.recall(question, top_k=2)
    mem_block = ''
    if recalled:
        mem_block = '\n[历史参考]\n' + '\n'.join(f'- 之前问过 {r["query"]!r} 的回答: {r["answer"][:80]}' for r in recalled)
        if verbose: print(f'💭 召回历史: {len(recalled)} 条')

    tools_desc = '\n'.join(f'- {n}: {t["desc"]}' for n, t in tools.items())
    sys = f'你是助手。可用工具:\n{tools_desc}{mem_block}\n格式：Thought/Action/Action Input 或 Thought/Final Answer'
    messages = [{'role': 'system', 'content': sys}, {'role': 'user', 'content': question}]
    trace = []
    while True:
        llm_in = '\n'.join(m['content'] for m in messages)
        out = llm(messages)
        ok, reason = budget.tick(llm_in, out)
        trace.append({'step': budget.iter, 'llm_out': out[:150]})
        if verbose: print(f'\n── step {budget.iter} ──\n{out[:150]}')
        if not ok:
            if verbose: print(f'⛔ 预算耗尽: {reason}')
            return {'answer': f'(中止: {reason}) ' + out[:80], 'trace': trace, 'budget': budget.report()}

        parsed = parse(out)
        if parsed['type'] == 'final':
            memory.remember(question, parsed['answer'])
            return {'answer': parsed['answer'], 'trace': trace, 'budget': budget.report()}
        if parsed['type'] == 'error':
            return {'answer': '(解析失败)', 'trace': trace, 'budget': budget.report()}
        # action
        name = parsed['tool']
        if name not in tools:
            obs = f'ERROR: unknown tool {name!r}'
        else:
            try: obs = tools[name]['fn'](**parsed['args'])
            except TypeError as e: obs = f'ERROR: bad args: {e}'
        trace[-1]['action'] = parsed; trace[-1]['observation'] = obs[:120]
        if verbose: print(f'TOOL[{name}]({parsed["args"]})> {obs[:120]}')
        messages.append({'role': 'assistant', 'content': out + f'\nObservation: {obs}'})

In [6]:
# 跑 4 个 query，看记忆累积 + 预算消耗
queries = [
    '帮我搜一下 python',
    '读 notes.txt 的内容',
    '把内容: 学习日志 写入到 log.txt',
    '帮我搜一下 python',   # 复问 —— 应该看到「历史参考」召回
]
for q in queries:
    print(f'\n{"="*60}\n📝 Q: {q}')
    budget = Budget(max_iter=6, token_budget=2000, time_budget_s=10)
    result = run_agent_v2(q, TOOLS, budget, memory, verbose=True)
    print(f'\n✅ Answer: {result["answer"][:150]}')
    print(f'📊 Budget: {result["budget"]}')

print(f'\n\n长期记忆累积: {len(memory.records)} 条')


📝 Q: 帮我搜一下 python

── step 1 ──
Thought: 搜 web
Action: search_web
Action Input: {"query": "python"}
TOOL[search_web]({'query': 'python'})> {"query": "python", "results": ["Python 是动态类型的解释型语言", "Python 创始人 Guido van Rossum"]}

── step 2 ──
Thought: 已得到结果
Final Answer: {"query": "python", "results": ["Python 是动态类型的解释型语言", "Python 创始人 Guido van Rossum"]}

✅ Answer: {"query": "python", "results": ["Python 是动态类型的解释型语言", "Python 创始人 Guido van Rossum"]}
📊 Budget: {'iter': 2, 'tokens_used': 322, 'time_s': 0.002}

📝 Q: 读 notes.txt 的内容

── step 1 ──
Thought: 读文件
Action: read_file
Action Input: {"path": "notes.txt"}
TOOL[read_file]({'path': 'notes.txt'})> 已存的笔记：
- python
- rag


── step 2 ──
Thought: 已得到结果
Final Answer: 已存的笔记：

✅ Answer: 已存的笔记：
📊 Budget: {'iter': 2, 'tokens_used': 276, 'time_s': 0.001}

📝 Q: 把内容: 学习日志 写入到 log.txt

── step 1 ──
Thought: 写入文件
Action: write_file
Action Input: {"path": "log.txt", "content": "学习日志 写入"}
TOOL[write_file]({'path': 'log.txt', 'content': '学习日志 写入'})> {"ok

## 5. 预算耗尽的故意演示

In [7]:
# 把 token_budget 设很小，强制超额
tight_budget = Budget(max_iter=5, token_budget=80, time_budget_s=10)
result = run_agent_v2('搜 python 然后写到 out.txt', TOOLS, tight_budget, memory, verbose=False)
print(f'结果: {result["answer"][:100]}')
print(f'预算报告: {result["budget"]}')
print('\n→ 生产里：预算耗尽时应该触发降级（如：返回部分结果、转人工、降模型）')

结果: (中止: 达到 token_budget=80（已用 124）) Thought: 搜 web
Action: search_web
Action Input: {"query": "python"}
预算报告: {'iter': 1, 'tokens_used': 124, 'time_s': 0.0}

→ 生产里：预算耗尽时应该触发降级（如：返回部分结果、转人工、降模型）


In [8]:
# 清理沙箱
import shutil
shutil.rmtree(SANDBOX, ignore_errors=True)
print('沙箱已清理')

沙箱已清理


## 深入思考

1. **为什么自动召回历史，不让 LLM 决定要不要 recall？**
   - LLM 可能忘了调，或调用方式不对。**「便宜的事情系统强制做」**比「相信 LLM 每次都做对」稳得多。
2. **长期记忆放向量库还是 KV？**
   - 向量库适合「按语义相似召回」（用户问类似问题 → 召回类似历史）；KV 适合「按 user_id / session_id 精确取」。**生产常常两者都有**。
3. **预算超了应该 hard stop 还是 soft warn？**
   - 软警告 → 让 LLM 知道「还剩 X 步、X token，请总结收尾」往往比硬停效果好。本 notebook 用 hard，生产推荐 soft + hard 两道。
4. **token 估算这么粗暴 (`zh + en/4`) 有用吗？**
   - 估个量级够用。真实计费按 tokenizer.encode 长度，但估算只用来「**避免远超预算**」，不要求精确。
5. **多工具时怎么避免 LLM 总选错工具？**
   - (a) 减少工具数（每场景 ≤ 8 个）(b) 工具描述写清「何时用、何时不用」(c) 提供典型示例 (d) 用 LLM-judge eval 跑工具选择正确率。

**改一改**：
- 复跑同样的 4 个 query，看每次 `召回历史: N 条` 数字怎么变
- 加第 5 个工具（如 mock `send_email`），看 LLM stub 怎么处理（OFFLINE 启发式不认识 → 会答不上来）

## 自检 ✅

- [ ] 默写工具设计 4 原则（小而专 / 可观测 / 可幂等 / 错误明确）
- [ ] 解释「长期记忆 vs RAG」的关系
- [ ] 解释 3 种预算（步数 / token / 时间）各自防什么
- [ ] 给一个工具列表 20 个的 Agent，能立刻指出「工具太多 → 召回错乱」的风险
- [ ] 解释「为什么系统强制 recall 比让 LLM 决定更稳」

## 下一步

→ [`29_minimal_mcp_server.ipynb`](29_minimal_mcp_server.ipynb)